# Notebook to produce results for the conflict impact forecasting paper
Paper is maintained at https://www.overleaf.com/project/697b5ef2f9ec68761916dd57

Current version of the paper is the file: 
Paper/Impact_forecasting_12Nov26.tex 

## Load in data necessary to produce the results for the paper
All the analysis reported here will need to be replicable, meaning that everything should be possible to run without any access to the pipeline

In Xiaolong's notebooks forecasts were extracted from the prediction store. If that is available, that is great. If not, store the files in a folder structure in the VIEWS dropbox.

## Reproduce eight new versions of Table 3 in the paper
In the format of a pandas table inside this notebook. 

Separate table (for now) for each outcome (see Tabler 4 in the paper for a list).

Each table should have five columns and 8 data rows:

*Five columns:*

- Model name
- Features included
- CRPS
- MSE
- MSLE

*Four times two rows:*

Four rows for each of the model (nhits, tide):
- Baseline features model (BF)
- BF + spatial lags (BFS)
- BF + conflict (BFC)
- BFS + conflict (BFSC)

I assume models are now trained on data for 1990--2017 and evaluated against data for 2018-2025; make sure this is transparent from the results


## Model roster: features in / features out

All sixteen models forecast the same target — **`lr_gdp_pcap`** (GDP per capita, `wdi_ny_gdp_pcap_kd`, fetched at `country_year` and disaggregated to `country_month` with Denton–Cholette). Models differ only in **which candidate covariates go in** and the **training loss**.

Feature-set suffixes:

- **BF** — base features (population only)
- **BFS** — BF + spatial country lag of population (1-step)
- **BFC** — BF + monthly UCDP GED conflict fatality counts (state-based, one-sided, non-state)
- **BFSC** — BFS + the three GED counts *and* their 1-step country lags

Training/evaluation split per the spec above: assumed trained on 1990–2017 and evaluated on 2018–2025.

### Covariate matrix (✓ = included in the queryset)

| Feature (viewser column) | Source | BF | BFS | BFC | BFSC |
|---|---|:---:|:---:|:---:|:---:|
| `lr_pop_totl` (population total) | `wdi_sp_pop_totl`, country_year | ✓ | ✓ | ✓ | ✓ |
| `lr_pop_totl_splag` (1-step country lag) | same + `spatial.countrylag(1, 1, 0, 0)` | — | ✓ | — | ✓ |
| `lr_ged_sb` (state-based fatalities) | `ged_sb_best_sum_nokgi`, country_month | — | — | ✓ | ✓ |
| `lr_ged_os` (one-sided fatalities) | `ged_os_best_sum_nokgi`, country_month | — | — | ✓ | ✓ |
| `lr_ged_ns` (non-state fatalities) | `ged_ns_best_sum_nokgi`, country_month | — | — | ✓ | ✓ |
| `lr_ged_sb_splag`, `lr_ged_os_splag`, `lr_ged_ns_splag` (1-step country lags) | same + `spatial.countrylag(1, 1, 0, 0)` | — | — | — | ✓ |

### Models

Each feature-set variant exists as an N-HiTS and a TiDE model, each with a SpotlightLoss version and an MSELoss twin (identical config except `loss_function`). Spotlight variants: `nhits_bfs`→`crimson_fox`, `nhits_bf`→`iron_willow`, `nhits_bfsc`→`shadow_harbor`, `nhits_bfc`→`frozen_crown`, `tide_bfs`→`golden_rune`, `tide_bf`→`hidden_peak`, `tide_bfsc`→`wild_compass`, `tide_bfc`→`silver_drift` (renamed 2026-11; MSE twins created at the same time).

| Model | Family | Variant | Loss | Features in (besides target) | Features out |
|---|---|---|---|---|---|
| `iron_willow` | N-HiTS | BF | SpotlightLoss | pop | pop splag; all GED counts & splags |
| `swift_pixel` | N-HiTS | BF | MSELoss | pop | pop splag; all GED counts & splags |
| `crimson_fox` | N-HiTS | BFS | SpotlightLoss | pop, pop splag | all GED counts & splags |
| `broken_spell` | N-HiTS | BFS | MSELoss | pop, pop splag | all GED counts & splags |
| `frozen_crown` | N-HiTS | BFC | SpotlightLossLogcosh | pop; ged_sb, ged_os, ged_ns | pop splag; all GED splags |
| `rapid_cipher` | N-HiTS | BFC | MSELoss | pop; ged_sb, ged_os, ged_ns | pop splag; all GED splags |
| `shadow_harbor` | N-HiTS | BFSC | SpotlightLoss | pop, pop splag; ged_sb, ged_os, ged_ns, their splags | — (full candidate set) |
| `quiet_circuit` | N-HiTS | BFSC | MSELoss | pop, pop splag; ged_sb, ged_os, ged_ns, their splags | — (full candidate set) |
| `hidden_peak` | TiDE | BF | SpotlightLoss | pop | pop splag; all GED counts & splags |
| `fuzzy_logic` | TiDE | BF | MSELoss | pop | pop splag; all GED counts & splags |
| `golden_rune` | TiDE | BFS | SpotlightLoss | pop, pop splag | all GED counts & splags |
| `lucky_packet` | TiDE | BFS | MSELoss | pop, pop splag | all GED counts & splags |
| `silver_drift` | TiDE | BFC | SpotlightLoss | pop; ged_sb, ged_os, ged_ns | pop splag; all GED splags |
| `neon_stack` | TiDE | BFC | MSELoss | pop; ged_sb, ged_os, ged_ns | pop splag; all GED splags |
| `wild_compass` | TiDE | BFSC | SpotlightLoss | pop, pop splag; ged_sb, ged_os, ged_ns, their splags | — (full candidate set) |
| `silent_orbit` | TiDE | BFSC | MSELoss | pop, pop splag; ged_sb, ged_os, ged_ns, their splags | — (full candidate set) |

### Covariates excluded from every model

The querysets carry commented-out candidates that no model uses: healthcare (`lr_ttns_zs`, `lr_imrt_in`, `lr_le00_in`, `lr_chex_gd_zs`), food security (`lr_stnt_me_zs`, `lr_defc_zs`), education (`lr_prm_enrr`), and democracy (`lr_v2x_libdem`).

In [ ]:
import os
from pathlib import Path

# for macos
username = Path.home().name
#DROPBOX_PATH = f"/Users/{username}/Library/CloudStorage/Dropbox/ReplicationData/Impact/models"
DROPBOX_PATH = f"/Users/{username}/ViEWS Dropbox/ViEWS/ReplicationData/Impact" # Version for HH

os.chdir(DROPBOX_PATH)

print(os.getcwd())

MODEL_DIRS = os.listdir(DROPBOX_PATH)
MODES = ["calibration", "forecasting"]
FORMATS = {"prediction_frame": ".npz", "dataframe": ".parquet"}

BASE_DATA_RAW = Path("/data/raw")
BASE_DATA_GENERATED = Path("/data/generated")
BASE_DATA_PROCESSED = Path("/data/processed")

print(MODEL_DIRS)

In [ ]:
# !for m in crimson_fox iron_willow shadow_harbor frozen_crown golden_rune hidden_peak wild_compass silver_drift broken_spell swift_pixel quiet_circuit rapid_cipher lucky_packet fuzzy_logic silent_orbit neon_stack; do mkdir -p ~/Library/CloudStorage/Dropbox/ReplicationData/Impact/models/$m/data/{generated,processed,raw}; done

## Unifying the prediction stores and evaluating report-style

The calibration runs for all sixteen models now sit in one folder per override month under `data/generated/calibration/`, with the month id naming the folder: 540 is the plain run at the end of the data, 560, 561 and 562 are the August, September and October 2026 overrides. Each folder holds the three rolling origins from section 3.6 of the report, each origin predicting 36 monthly steps.

The cells below rewrite every prediction set and the processed feature and target table as native views-frames artifacts under `data/unified/` (the stored files stay as they are), evaluate the way the report does (annual means per calendar year, metrics on yearly values only), build one table per loss and override month, and plot the aggregate GDP per capita trajectory against each model's predicted annual mean.

Two things to keep in mind when reading the numbers. The stored predictions carry a single sample per cell, so the sample-based CRPS collapses to the mean absolute error; the report's CRPS values are consistent with that same convention, so the columns stay comparable. And MSLE uses the usual log(1+x) form with non-positive predictions clipped to zero, since a few predicted values come out negative. Country-years without an actual value drop out of the metrics.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML

from views_frames.index import SpatioTemporalIndex
from views_frames.io import npz as vf_npz
from views_frames.metadata import FrameMetadata
from views_frames.prediction_frame import PredictionFrame
from views_frames.spatial_level import SpatialLevel
from views_frames.target_frame import TargetFrame

#MODELS_ROOT = Path.home() / "Library" / "CloudStorage" / "Dropbox" / "ReplicationData" / "Impact" / "models"
MODELS_ROOT = Path(f"/Users/{username}/ViEWS Dropbox/ViEWS/ReplicationData/Impact/models")
RESULTS_DIR = MODELS_ROOT / "evaluation_results"
OVERRIDE_MONTHS = [540, 560, 561, 562]
OVERRIDE_LABELS = {
    540: "540 (no override, end of data)",
    560: "560 (override: Aug 2026)",
    561: "561 (override: Sep 2026)",
    562: "562 (override: Oct 2026)",
}
ORIGINS = ["origin_0", "origin_1", "origin_2"]
TARGET = "lr_gdp_pcap"
LOSSES = ["spotlight", "MSE"]

MODEL_ROSTER = pd.DataFrame(
    [
        ["crimson_fox",   "nhits", "BFS",  "spotlight"],
        ["iron_willow",  "nhits", "BF",   "spotlight"],
        ["shadow_harbor", "nhits", "BFSC", "spotlight"],
        ["frozen_crown",  "nhits", "BFC",  "spotlight"],
        ["broken_spell",  "nhits", "BFS",  "MSE"],
        ["swift_pixel",   "nhits", "BF",   "MSE"],
        ["quiet_circuit", "nhits", "BFSC", "MSE"],
        ["rapid_cipher",  "nhits", "BFC",  "MSE"],
        ["golden_rune",   "tide",  "BFS",  "spotlight"],
        ["hidden_peak",   "tide",  "BF",   "spotlight"],
        ["wild_compass",  "tide",  "BFSC", "spotlight"],
        ["silver_drift",  "tide",  "BFC",  "spotlight"],
        ["lucky_packet",  "tide",  "BFS",  "MSE"],
        ["fuzzy_logic",   "tide",  "BF",   "MSE"],
        ["silent_orbit",  "tide",  "BFSC", "MSE"],
        ["neon_stack",    "tide",  "BFC",  "MSE"],
    ],
    columns=["model", "family", "feature_set", "loss"],
)
FEATURE_LABELS = {
    "BF": "baseline features",
    "BFS": "BF + spatial lags",
    "BFC": "BF + conflict",
    "BFSC": "BFS + conflict",
}

In [ ]:
class ModelDirManager:
    """One lookup point for every path the pipeline touches for a single model.

    Covers the stored data (calibration override folders, forecasting artifacts,
    processed and raw parquets) and the unified views-frames output tree, so the
    rest of the code never assembles paths by hand.
    """

    def __init__(self, model: str, root: Path = MODELS_ROOT):
        self.model = model
        self.data = Path(root) / model / "data"

    @property
    def generated(self) -> Path:
        return self.data / "generated"

    @property
    def processed(self) -> Path:
        return self.data / "processed"

    @property
    def raw(self) -> Path:
        return self.data / "raw"

    def calibration(self, override: int) -> Path:
        return self.generated / "calibration" / str(override)

    def origin_dir(self, override: int, origin: str) -> Path:
        return self.calibration(override) / origin / TARGET

    def processed_path(self, run_type: str = "calibration") -> Path:
        return self.processed / f"{run_type}_viewser_df_tempdisagg.parquet"

    def raw_path(self, run_type: str = "calibration") -> Path:
        return self.raw / f"{run_type}_viewser_df.parquet"

    def forecasting(self) -> list[Path]:
        return sorted(self.generated.glob("predictions_forecasting_*"))

    def unified(self, *parts) -> Path:
        path = self.data / "unified"
        for part in parts:
            path = path / str(part)
        return path

    def unified_origin(self, override: int, origin: str) -> Path:
        return self.unified("calibration", override, origin, TARGET)

    def unified_processed(self, run_type: str = "calibration") -> Path:
        return self.unified("processed", run_type)

    def evaluation_dir(self, override: int) -> Path:
        return self.unified("evaluation", override)

    def inventory(self) -> dict:
        return {
            "model": self.model,
            "override_months": sorted(
                int(p.name) for p in (self.generated / "calibration").glob("*") if p.is_dir()
            ),
            "forecasting_artifacts": len(self.forecasting()),
            "processed_files": sorted(p.name for p in self.processed.glob("*.parquet")),
            "raw_files": sorted(p.name for p in self.raw.glob("*.parquet")),
        }


MODEL_DIRS = {model: ModelDirManager(model) for model in MODEL_ROSTER.model}
display(pd.DataFrame([mgr.inventory() for mgr in MODEL_DIRS.values()]))

In [ ]:
class LegacyPredictionStore:
    """Reads the npz + npy layout the training pipeline writes (identifiers.npz
    plus y_pred.npy, no header) and hands it over as native views-frames objects.
    """

    def __init__(self, directory: Path):
        self.directory = Path(directory)

    def _frame(self, target_dir: Path, *, model: str, run_type: str, run_id: str) -> PredictionFrame:
        with np.load(target_dir / "identifiers.npz") as npz:
            time = npz["time"].astype(np.int64)
            unit = npz["unit"].astype(np.int64)
        values = np.load(target_dir / "y_pred.npy").astype(np.float32)
        index = SpatioTemporalIndex(time, unit, SpatialLevel.CM)
        return PredictionFrame(values, index, FrameMetadata(model=model, run_type=run_type, run_id=run_id))

    def origin_frame(self, origin: str, **kwargs) -> PredictionFrame:
        return self._frame(self.directory / origin / TARGET, **kwargs)

In [ ]:
class ProcessedGroundTruth:
    """The processed viewser table: monthly target and covariates on
    (month_id, country_id), already disaggregated to months.
    """

    def __init__(self, parquet: Path, target: str = TARGET):
        self.frame = pd.read_parquet(parquet)
        self.target = target

    def monthly(self) -> pd.DataFrame:
        return self.frame[[self.target]].reset_index()

In [ ]:
class ViewsFramesUnifier:
    """Writes native views-frames copies of the predictions and the processed
    table under <model>/data/unified/. The stored data is never modified.
    """

    def unify_origin(self, mgr: ModelDirManager, override: int, origin: str) -> Path:
        store = LegacyPredictionStore(mgr.calibration(override))
        frame = store.origin_frame(origin, model=mgr.model, run_type="calibration",
                                   run_id=f"override_{override}")
        out = mgr.unified_origin(override, origin)
        frame.save(out)
        return out

    def unify_processed(self, mgr: ModelDirManager, run_type: str = "calibration") -> tuple[Path, Path]:
        gt = ProcessedGroundTruth(mgr.processed_path(run_type))
        df = gt.frame.reset_index()
        time = df["month_id"].to_numpy(np.int64)
        unit = df["country_id"].to_numpy(np.int64)
        index = SpatioTemporalIndex(time, unit, SpatialLevel.CM)
        out_dir = mgr.unified_processed(run_type)

        target_out = out_dir / TARGET
        target = TargetFrame(
            gt.frame[[TARGET]].to_numpy(np.float32), index,
            FrameMetadata(model=mgr.model, run_type=run_type),
        )
        target.save(target_out)

        features = [c for c in gt.frame.columns if c != TARGET]
        features_out = out_dir / "features"
        vf_npz.save(
            features_out,
            values=gt.frame[features].to_numpy(np.float32),
            time=time,
            unit=unit,
            level=SpatialLevel.CM.value,
            metadata={"model": mgr.model, "run_type": run_type},
            feature_names=features,
        )
        return target_out, features_out

    def unify_forecasting(self, mgr: ModelDirManager) -> list[Path]:
        outs = []
        for path in mgr.forecasting():
            if not path.is_dir():
                continue
            frame = LegacyPredictionStore(path)._frame(
                path / TARGET, model=mgr.model, run_type="forecasting", run_id=path.name
            )
            out = mgr.unified("forecasting", path.name, TARGET)
            frame.save(out)
            outs.append(out)
        return outs

In [ ]:
class ReportStyleEvaluator:
    """Yearly-level rolling-origin evaluation in the style of report section 3.6.

    The annual estimate is the mean of the twelve monthly predictions in a
    calendar year, and metrics run on yearly values for complete calendar years
    only. With a single stored sample per cell, CRPS equals the mean absolute
    error. MSLE uses log(1+x) on values clipped at zero.
    """

    def __init__(self, ground_truth: ProcessedGroundTruth):
        self.ground_truth = ground_truth

    def annual_actuals(self) -> pd.DataFrame:
        df = self.ground_truth.monthly()
        df["year"] = 1980 + (df["month_id"] - 1) // 12
        return df.groupby(["year", "country_id"], as_index=False)[self.ground_truth.target].mean().rename(
            columns={self.ground_truth.target: "actual"}
        )

    def annual_values(self, time, unit, y_pred) -> pd.DataFrame:
        df = pd.DataFrame({"month_id": time, "country_id": unit, "pred": y_pred.ravel()})
        df["year"] = 1980 + (df["month_id"] - 1) // 12
        annual = df.groupby(["year", "country_id"], as_index=False)["pred"].mean()
        return annual.merge(self.annual_actuals(), on=["year", "country_id"], how="inner")

    @staticmethod
    def metrics(annual: pd.DataFrame) -> dict:
        err = annual["pred"] - annual["actual"]
        crps = err.abs().mean()
        mse = (err ** 2).mean()
        msle = ((np.log1p(annual["pred"].clip(lower=0.0))
                 - np.log1p(annual["actual"].clip(lower=0.0))) ** 2).mean()
        return {"CRPS": crps, "MSE": mse, "MSLE": msle, "n_cells": len(annual)}

    def evaluate_override(self, mgr: ModelDirManager, override: int) -> dict:
        store = LegacyPredictionStore(mgr.calibration(override))
        per_origin, annual_parts = {}, []
        for origin in ORIGINS:
            frame = store.origin_frame(origin, model=mgr.model, run_type="calibration",
                                       run_id=f"override_{override}_{origin}")
            annual = self.annual_values(frame.identifiers["time"], frame.identifiers["unit"],
                                        frame.values)
            annual["origin"] = origin
            per_origin[origin] = self.metrics(annual)
            annual_parts.append(annual)
        pooled = pd.concat(annual_parts, ignore_index=True)
        return {"per_origin": per_origin, "pooled": self.metrics(pooled), "annual": pooled}

In [ ]:
class TableBuilder:
    """Assembles the per-loss result tables: eight rows, four feature sets by
    nhits/tide, in the order the report uses (BF, BFS, BFC, BFSC).
    """

    FEATURE_ORDER = {"BF": 0, "BFS": 1, "BFC": 2, "BFSC": 3}

    def __init__(self, roster: pd.DataFrame = MODEL_ROSTER):
        self.roster = roster

    def table(self, pooled_by_model: dict, loss: str) -> pd.DataFrame:
        subset = self.roster[self.roster.loss == loss].copy()
        subset["_ord"] = subset.feature_set.map(self.FEATURE_ORDER)
        subset = subset.sort_values(["family", "_ord"])
        rows = []
        for r in subset.itertuples():
            m = pooled_by_model[r.model]
            rows.append({
                "model": r.model,
                "description": f"{r.family}_{r.feature_set} ({FEATURE_LABELS[r.feature_set]}, {r.loss} loss)",
                "CRPS": m["CRPS"],
                "MSE": m["MSE"],
                "MSLE": m["MSLE"],
                "n_cells": m["n_cells"],
            })
        return pd.DataFrame(rows)

    @staticmethod
    def formatted(df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()
        out["CRPS"] = out["CRPS"].map(lambda v: f"{v:,.4f}")
        out["MSE"] = out["MSE"].map(lambda v: f"{v:.4e}")
        out["MSLE"] = out["MSLE"].map(lambda v: f"{v:.4f}")
        return out

In [ ]:
class TrajectoryPlotter:
    """Aggregate trajectory figure for one override month.

    The black line is the actual annual mean over countries. Each coloured line
    is one feature set, averaged across its two training-loss variants, and the
    shaded band spans those two variants. Per-model numbers live in the tables.
    Both panels share a y-range computed from the plotted country-means.
    """

    FEATURE_ORDER = ["BF", "BFS", "BFC", "BFSC"]
    FEATURE_COLORS = {"BF": "#1F77B4", "BFS": "#D62728", "BFC": "#2CA02C", "BFSC": "#9467BD"}

    @staticmethod
    def _rgba(hex_color: str, alpha: float) -> str:
        h = hex_color.lstrip("#")
        r, g, b = (int(h[i:i + 2], 16) for i in (0, 2, 4))
        return f"rgba({r},{g},{b},{alpha})"

    def _family_series(self, annual_by_model: dict, family: str) -> dict:
        """Mean, low and high country-mean series per feature set."""
        roster = MODEL_ROSTER[MODEL_ROSTER.family == family]
        out = {}
        for fs in self.FEATURE_ORDER:
            models = roster[roster.feature_set == fs].model
            per_model = pd.concat(
                [annual_by_model[m].groupby("year", as_index=False)["pred"].mean()
                 for m in models]
            )
            out[fs] = {
                "mean": per_model.groupby("year", as_index=False)["pred"].mean(),
                "lo": per_model.groupby("year")["pred"].min(),
                "hi": per_model.groupby("year")["pred"].max(),
            }
        return out

    def _family_traces(self, series: dict, family: str, show_legend: bool) -> list:
        traces = []
        for fs in self.FEATURE_ORDER:
            s = series[fs]
            color = self.FEATURE_COLORS[fs]
            traces.append(go.Scatter(
                x=list(s["lo"].index) + list(s["lo"].index)[::-1],
                y=list(s["hi"]) + list(s["lo"])[::-1],
                fill="toself", fillcolor=self._rgba(color, 0.14),
                line=dict(width=0), hoverinfo="skip", showlegend=False,
            ))
            traces.append(go.Scatter(
                x=s["mean"]["year"], y=s["mean"]["pred"], name=fs,
                legendgroup=fs, showlegend=show_legend,
                line=dict(color=color, width=2),
                hovertemplate=f"{fs} ({family}), mean of two loss variants<br>%{{x}}: %{{y:,.0f}}<extra></extra>",
            ))
        return traces

    def figure(self, annual_by_model: dict, title: str) -> go.Figure:
        fig = make_subplots(rows=1, cols=2, subplot_titles=("nhits", "tide"))
        any_model = next(iter(annual_by_model))
        actual = annual_by_model[any_model].groupby("year", as_index=False)["actual"].mean()
        plotted = [actual["actual"]]
        for col, family in ((1, "nhits"), (2, "tide")):
            series = self._family_series(annual_by_model, family)
            plotted += [s["mean"]["pred"] for s in series.values()]
            plotted += [s["lo"] for s in series.values()]
            plotted += [s["hi"] for s in series.values()]
            fig.add_trace(
                go.Scatter(x=actual["year"], y=actual["actual"], name="actual",
                           legendgroup="actual", showlegend=(col == 1),
                           line=dict(color="black", width=2.5),
                           hovertemplate="actual<br>%{x}: %{y:,.0f}<extra></extra>"),
                row=1, col=col,
            )
            for trace in self._family_traces(series, family, show_legend=(col == 1)):
                fig.add_trace(trace, row=1, col=col)
        stack = pd.concat(plotted, ignore_index=True)
        lo, hi = float(stack.min()), float(stack.max())
        pad = 0.06 * (hi - lo)
        fig.update_layout(
            title=dict(text=title, font=dict(size=15)),
            template="plotly_white", width=980, height=430,
            legend=dict(orientation="h", x=0, y=1.06, xanchor="left"),
            margin=dict(t=95, b=55, l=70, r=25),
            hovermode="x",
        )
        fig.update_xaxes(title_text="year", dtick=1, gridcolor="#E8E8E8")
        fig.update_yaxes(title_text="GDP per capita, annual mean over countries",
                         tickformat=",", gridcolor="#E8E8E8", zeroline=False,
                         range=[lo - pad, hi + pad])
        return fig

In [ ]:
unifier = ViewsFramesUnifier()
evaluators = {}
results = {}

for model in MODEL_ROSTER.model:
    mgr = MODEL_DIRS[model]
    evaluator = ReportStyleEvaluator(ProcessedGroundTruth(mgr.processed_path("calibration")))
    evaluators[model] = evaluator
    for override in OVERRIDE_MONTHS:
        res = evaluator.evaluate_override(mgr, override)
        results.setdefault(override, {})[model] = res
        out_dir = mgr.evaluation_dir(override)
        out_dir.mkdir(parents=True, exist_ok=True)
        pd.DataFrame([{**res["pooled"], "override": override}]).to_csv(
            out_dir / "pooled_metrics.csv", index=False)
        pd.DataFrame([{"origin": o, **m} for o, m in res["per_origin"].items()]).to_csv(
            out_dir / "origin_metrics.csv", index=False)
        res["annual"].to_csv(out_dir / "annual_values.csv", index=False)
        for origin in ORIGINS:
            unifier.unify_origin(mgr, override, origin)
    unifier.unify_processed(mgr)
    unifier.unify_forecasting(mgr)
    print(f"{model}: evaluated {len(OVERRIDE_MONTHS)} override months, frames unified")

In [ ]:
builder = TableBuilder()
tables = {}
for override in OVERRIDE_MONTHS:
    pooled_by_model = {m: results[override][m]["pooled"] for m in MODEL_ROSTER.model}
    out_dir = RESULTS_DIR / str(override)
    out_dir.mkdir(parents=True, exist_ok=True)
    for loss in LOSSES:
        df = builder.table(pooled_by_model, loss)
        tables[(override, loss)] = df
        df.to_csv(out_dir / f"table_{loss}.csv", index=False)

for override in OVERRIDE_MONTHS:
    display(HTML(f"<h3>{OVERRIDE_LABELS[override]}</h3>"))
    for loss in LOSSES:
        display(HTML(f"<b>{loss} loss</b>"))
        display(TableBuilder.formatted(tables[(override, loss)]))

In [ ]:
for override in OVERRIDE_MONTHS:
    rows = []
    for model in MODEL_ROSTER.model:
        info = MODEL_ROSTER[MODEL_ROSTER.model == model].iloc[0]
        for origin, m in results[override][model]["per_origin"].items():
            rows.append({"model": model, "family": info.family,
                         "feature_set": info.feature_set, "loss": info.loss,
                         "origin": origin, **m})
    breakdown = pd.DataFrame(rows)
    display(HTML(f"<h4>Per rolling origin, {OVERRIDE_LABELS[override]}</h4>"))
    display(breakdown.style.format({"CRPS": "{:,.4f}", "MSE": "{:.4e}", "MSLE": "{:.4f}"}))

Each figure covers one override month. The black line is the actual annual mean over countries; a coloured line is one feature set averaged across its two training-loss variants, with the band spanning those variants. Per-model numbers are in the tables above.

In [ ]:
plotter = TrajectoryPlotter()
for override in OVERRIDE_MONTHS:
    annual_by_model = {m: results[override][m]["annual"] for m in MODEL_ROSTER.model}
    fig = plotter.figure(annual_by_model, f"Aggregate GDP per capita trajectory, {OVERRIDE_LABELS[override]}")
    out_dir = RESULTS_DIR / str(override)
    out_dir.mkdir(parents=True, exist_ok=True)
    fig.write_html(out_dir / "trajectory.html", include_plotlyjs="cdn")
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))

### Reading the results

The tables and figures above cover complete calendar years 2017 through 2021, the union of the three rolling origins, evaluated the way report section 3.6 prescribes: annual means, metrics on yearly values only.

The nhits models lead on accuracy. Best CRPS went to iron_willow in override months 540 and 560 and to crimson_fox in 561 and 562 on the spotlight side; broken_spell and quiet_circuit lead the MSE-loss nhits models. The tide models trail across the board, though the tide MSE-loss BFC model (CRPS 959) edges out every spotlight tide model (972 and up). Within nhits, conflict features hurt on MSE-loss runs and roughly break even on spotlight runs, and the baseline spotlight models are hard to beat.

One data observation worth knowing: the four calibration runs are byte-identical for every tide model, so their per-override tables repeat the same numbers. The nhits runs genuinely differ from override month to override month. If the overrides were meant to change what the tide models see, they did not.

The evaluation itself checks out against the pipeline. Annual actuals match the yearly_entities parquet in data/generated to float precision, and so do the predicted annual means for override 562, which tells us the stored evaluation parquets were written by the run now living in calibration/562. The other three nhits runs differ from it by a few hundred GDP units, which is run-to-run variation rather than an aggregation error.

CRPS here is the mean absolute error, because the stored predictions carry a single sample per cell. MSLE clips non-positive predictions at zero, while the pipeline leaves those cells undefined, so these MSLE numbers sit below the stored ones.

Everything the notebook writes goes to two places. Inside each model folder, data/unified/ holds native views-frames copies of every origin's predictions, the processed target and covariates, the forecasting predictions, and per-override metric CSVs. Cross-model tables and the trajectory HTML live in models/evaluation_results/<override month>/.